# 08 — Silver integrado: merge meteo + socio

Cruza clima semanal con variables sociodemográficas anuales por `distrito_key` + `anio`. Salida: `data/silver/integrado/meteo_socio_piura_2017_2025.csv`.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

In [2]:
import pandas as pd
from src.utils.paths import SILVER, SILVER_METEO_SOCIO
from src.processing.merge_datasets import diagnosticar_distritos_sin_match, merge_meteo_socio

meteo = pd.read_csv(SILVER / "meteo_semanal_distrital.csv")
socio_anual = pd.read_csv(SILVER / "socio_anual.csv", dtype={"ubigeo": "string"})


In [3]:
diagnosticar_distritos_sin_match(meteo, socio_anual)

En meteo pero no en socio: {'EL CARMENDELA FRONTERA', 'SAN MIGUELDE EL FAIQUE'}
En socio pero no en meteo: {'SAN MIGUEL DE EL FAIQUE', 'EL CARMEN DE LA FRONTERA'}


In [4]:
'''
Correr esta celda para corregir los nombres que no coinciden entre los datasets y luego volver a correr el merge. 
Cambiar con los nombres que salen que no coinciden. 
GADM siempre tiene algun nombre mal cada que se re ejecuta el fetch.
'''

from src.utils.keys import normalizar


meteo.loc[meteo["distrito"] == "El Carmendela Frontera", "distrito"] = "El Carmen de la Frontera"
meteo.loc[meteo["distrito"] == "San Miguelde El Faique", "distrito"] = "San Miguel de el Faique"


meteo["distrito_key"] = meteo["distrito"].apply(normalizar)



In [5]:
diagnosticar_distritos_sin_match(meteo, socio_anual)

En meteo pero no en socio: set()
En socio pero no en meteo: set()


In [6]:
socio_anual.info()

<class 'pandas.DataFrame'>
RangeIndex: 585 entries, 0 to 584
Data columns (total 22 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   ubigeo                          585 non-null    string 
 1   departamento                    585 non-null    str    
 2   provincia                       585 non-null    str    
 3   distrito                        585 non-null    str    
 4   distrito_key                    585 non-null    str    
 5   anio                            585 non-null    int64  
 6   poblacion                       585 non-null    int64  
 7   fraccion_rural                  585 non-null    float64
 8   fraccion_mujeres                585 non-null    float64
 9   fraccion_menores_15             585 non-null    float64
 10  fraccion_sin_seguro             585 non-null    float64
 11  fraccion_analfabeta_15_mas      585 non-null    float64
 12  fraccion_pared_precaria         585 non-null   

In [7]:
df_meteo_socio = merge_meteo_socio(meteo, socio_anual)
df_meteo_socio.info()

0 filas sin match sociodemográfico
<class 'pandas.DataFrame'>
RangeIndex: 30485 entries, 0 to 30484
Data columns (total 34 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   provincia                       30485 non-null  str    
 1   distrito                        30485 non-null  str    
 2   lat                             30485 non-null  float64
 3   lon                             30485 non-null  float64
 4   anio                            30485 non-null  int64  
 5   semana                          30485 non-null  int64  
 6   semana_inicio                   30485 non-null  str    
 7   temp_media                      30485 non-null  float64
 8   temp_max                        30485 non-null  float64
 9   temp_min                        30485 non-null  float64
 10  precip_total_mm                 30485 non-null  float64
 11  lluvia_total_mm                 30485 non-null  float64
 12  hum_rel_

In [8]:
SILVER_METEO_SOCIO.parent.mkdir(parents=True, exist_ok=True)
df_meteo_socio.to_csv(SILVER_METEO_SOCIO, index=False)